**Purpose:** PPO agent variant: + Reddit sentiment features.

**Outputs:** `DRLPPO_reddit_weights.npy`, `{prefix}_cum.png`, `{prefix}_weights.png`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.config import PROJECT_ROOT
from src.seeds import SEED_PORTFOLIO_DRL_PPO_V9_1_X_REDDIT, SEED_PORTFOLIO_DRL_PPO_V9_1_X_REDDIT_DEFAULT


# DRL PPO — Reddit Feature Extension (v9.1.3-reddit)

Extends **v9.1.3** by adding Reddit-based sentiment features per asset.

**Strategy**
- Load best trial from v9.1.3 Optuna DB → **freeze** its technical feature picks
- New Optuna search tunes **reddit params + all PPO / training hyperparameters**  
  (technical feature picks are kept fixed from the frozen best trial)
- Objective: mean Sharpe across 3 walk-forward CV folds × N seeds (same as v9.1.3)
- Final test: mid-test refit (same 2-phase pipeline as v9.1.3)

**Reddit search space**

| Parameter | Choices |
|---|---|
| `reddit_live_group` | `balance` \| `balance_ratio` \| `count_positive+count_negative` \| `ratio_positive+ratio_negative` |
| `reddit_hist_group` | `none` \| `balance_ratio_roll22` \| `balance_ratio_roll3` \| `balance_ratio_roll5` \| `balance_ratio_t-1+balance_ratio_t-5` \| `balance_ratio_variation1+balance_ratio_variation5` |
| `reddit_threshold`  | 0.00, 0.50, 0.75, 0.90 |
| `reddit_nan_strategy`    | zero_fill, carry_forward, mask_asset |

**Live group is mandatory** (always exactly one). **Historical group is optional** (`none` = skip).  
Paired indicators always enter together.

**`_REDDIT_NAN`** is a single global column (not per-asset) — 1 on days the feed was down.  
It is always appended as an `identity`-normalised feature, broadcast to all N assets.

**Sentiment column naming in df**
```
{ASSET}_reddit_threshold{THRESHOLD}_{INDICATOR}   ← per-asset sentiment values
_REDDIT_NAN                                        ← single global availability flag
```
e.g. `XLB_reddit_threshold0.50_balance_ratio`, `_REDDIT_NAN`


---
## 1 · Imports & Shared Infrastructure

In [ ]:
from __future__ import annotations

import copy
import json
import math
import os
import random
import time
import logging
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import gymnasium as gym
from gymnasium import spaces

import torch as th
from torch import nn

import optuna
from optuna.samplers import TPESampler
from optuna.trial import TrialState

from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback, CallbackList

import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)


---
## 2 · Config

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
DATASET_PATH  = str(PROJECT_ROOT / '03_portfolio/dataset.parquet')
SPREADS_PATH  = str(PROJECT_ROOT / '01_data/aux/bid-ask_spread.json')

# Path to the v9.1.3 Optuna DB — best trial's feature picks are frozen from here
V913_DB_PATH  = 'v913/ppo_stationary_all.db'
V913_STUDY_NAME = 'ppo_stationary_fixed_setup'

# ── Universe ──────────────────────────────────────────────────────────────
ASSETS = ['XLB', 'XLC', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV', 'XLY']

# ── Base feature buckets (used only to verify frozen picks) ───────────────
FEATURES = {
    'Price':      ['Close'],
    'OHLC':       ['Open', 'High', 'Low'],
    'Trend':      ['SMA5', 'SMA22', 'SMA63', 'SMA126', 'SMA252', 'MACD', 'ADX'],
    'Momentum':   ['RSI', 'MACD', 'CCI'],
    'Volume':     ['Volume', 'OBV', 'ADI', 'VolumeVariation'],
    'Volatility': ['BBUpper', 'BBLower', '5dayVolatility', '22dayVolatility',
                   '63dayVolatility', 'RatioVolatility'],
    'Market':     ['VIXIndexClose'],
}

# ── Reddit feature catalogue ───────────────────────────────────────────
# REDDIT_LIVE: exactly ONE group must be chosen (mandatory)
# Paired indicators are tuples — they always enter together.
REDDIT_LIVE_GROUPS = [
    'balance',
    'balance_ratio',
    ('count_positive', 'count_negative'),
    ('ratio_positive', 'ratio_negative'),
]

# REDDIT_HISTORICAL: at most ONE group (optional — 'none' skips)
REDDIT_HIST_GROUPS = [
    'none',
    'balance_ratio_roll22',
    'balance_ratio_roll3',
    'balance_ratio_roll5',
    ('balance_ratio_t-1', 'balance_ratio_t-5'),
    ('balance_ratio_variation1', 'balance_ratio_variation5'),
]

REDDIT_THRESHOLDS = ['0.00', '0.50', '0.75', '0.90']

# ── Study settings ─────────────────────────────────────────────────────────
OUT_DIR  = 'v913_reddit'
N_TRIALS = 0        # set > 0 to run the search
SEEDS    = [random.randint(0, 2**32 - 1) for _ in range(3)]


---
## 3 · Logging Helper

In [ ]:
def setup_logger(log_dir: str, name_prefix: str = 'ppo') -> logging.Logger:
    logger = logging.getLogger(f'{name_prefix}_{os.path.basename(log_dir)}')
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter('[%(asctime)s] [%(levelname)s] %(message)s')
    ch  = logging.StreamHandler()
    ch.setFormatter(fmt)
    logger.addHandler(ch)
    logger.propagate = False
    return logger


---
## 4 · Per-Asset Feature Extractor

Unchanged from v9.1.3 — the extractor is generic over `n_features`, so it
automatically handles the extra sentiment + `_REDDIT_NAN` dimensions added
by the new `build_raw_arrays_with_reddit`.


In [ ]:
class PerAssetFeaturesExtractor(BaseFeaturesExtractor):
    """
    Structured feature extractor that respects the per-asset layout.

    Observation layout (flat vector):
        [lookback * N * F]  per-asset feature windows  (F now includes sentiment + _REDDIT_NAN)
        [1]                 market signal (optional)
        [N]                 previous portfolio weights
        [N]                 asset availability mask
    """

    def __init__(
        self,
        observation_space,
        n_assets:      int,
        lookback:      int,
        n_features:    int,
        has_market:    bool,
        encoder_width: int = 64,
    ):
        self.n_assets      = n_assets
        self.lookback      = lookback
        self.n_features    = n_features
        self.has_market    = has_market
        self.encoder_width = encoder_width

        n_extra      = (1 if has_market else 0) + n_assets + n_assets
        features_dim = encoder_width + n_extra

        super().__init__(observation_space, features_dim=features_dim)

        asset_input_dim = lookback * n_features
        self.asset_encoder = nn.Sequential(
            nn.Linear(asset_input_dim, encoder_width),
            nn.Tanh(),
        )

    def forward(self, obs: th.Tensor) -> th.Tensor:
        B = obs.shape[0]
        N, L, F = self.n_assets, self.lookback, self.n_features

        asset_block_len = N * L * F
        asset_flat = obs[:, :asset_block_len]
        rest        = obs[:, asset_block_len:]

        asset_flat = asset_flat.view(B, N, L * F)
        asset_emb  = self.asset_encoder(asset_flat)
        pooled     = asset_emb.max(dim=1).values

        return th.cat([pooled, rest], dim=1)


---
## 5 · Walk-Forward & Train/Test Splits  *(unchanged from v9.1.3)*

In [ ]:
def make_walk_forward_splits(dates_index: pd.DatetimeIndex):
    splits = []
    periods = [
        (pd.Timestamp('2015-07-01'), pd.Timestamp('2020-06-30'),
         pd.Timestamp('2020-07-01'), pd.Timestamp('2021-06-30')),
        (pd.Timestamp('2015-07-01'), pd.Timestamp('2021-06-30'),
         pd.Timestamp('2021-07-01'), pd.Timestamp('2022-06-30')),
        (pd.Timestamp('2015-07-01'), pd.Timestamp('2022-06-30'),
         pd.Timestamp('2022-07-01'), pd.Timestamp('2023-06-30')),
    ]
    for tr_start, tr_end, v_start, v_end in periods:
        train_idx = np.where((dates_index >= tr_start) & (dates_index <= tr_end))[0]
        val_idx   = np.where((dates_index >= v_start)  & (dates_index <= v_end))[0]
        if len(train_idx) == 0 or len(val_idx) == 0:
            continue
        splits.append((train_idx, val_idx))
    if not splits:
        raise ValueError('No valid walk-forward splits.')
    return splits


def make_final_train_test_split(
    dates_index: pd.DatetimeIndex,
    train_start: str = '2015-07-01', train_end: str = '2023-06-30',
    test_start:  str = '2023-07-01', test_end:  str = '2025-06-30',
) -> Tuple[np.ndarray, np.ndarray]:
    ts = pd.Timestamp(train_start); te = pd.Timestamp(train_end)
    ss = pd.Timestamp(test_start);  se = pd.Timestamp(test_end)
    train_idx = np.where((dates_index >= ts) & (dates_index <= te))[0]
    test_idx  = np.where((dates_index >= ss) & (dates_index <= se))[0]
    if len(train_idx) == 0 or len(test_idx) == 0:
        raise ValueError('Invalid final split.')
    return train_idx, test_idx


def make_midtest_refit_split(
    test_idx: np.ndarray,
    dates_index: pd.DatetimeIndex,
    mid_date: str = '2024-07-01',
) -> Tuple[np.ndarray, np.ndarray]:
    mid = pd.Timestamp(mid_date)
    test_dates  = dates_index[test_idx]
    test_first  = test_idx[test_dates < mid]
    test_second = test_idx[test_dates >= mid]
    if len(test_first) == 0 or len(test_second) == 0:
        raise ValueError(f'Mid-test split at {mid_date} produced an empty half.')
    return test_first, test_second


def split_train_for_early_stop(
    train_idx: np.ndarray,
    dates_index: pd.DatetimeIndex,
    es_years: int = 1,
    min_train_days: int = 252,
) -> Tuple[np.ndarray, np.ndarray]:
    train_dates   = dates_index[train_idx]
    es_start_date = train_dates[-1] - pd.DateOffset(years=es_years) + pd.Timedelta(days=1)
    es_mask   = train_dates >= es_start_date
    core_idx  = train_idx[~es_mask]
    es_idx    = train_idx[es_mask]
    if len(core_idx) < min_train_days or len(es_idx) < min_train_days // 2:
        cut      = max(min_train_days, len(train_idx) - 252 * es_years)
        cut      = min(cut, len(train_idx) - 1)
        core_idx = train_idx[:cut]
        es_idx   = train_idx[cut:]
    if len(core_idx) == 0 or len(es_idx) == 0:
        raise ValueError('Failed to create non-empty early-stopping split.')
    return core_idx, es_idx


---
## 6 · Load Frozen Config from v9.1.3

Reads the best completed trial from the v9.1.3 Optuna DB and freezes:
- `feature_picks`  — which technical feature bucket to use per bucket
- `per_asset_feats` — flat ordered list of per-asset feature names
- `lookback`
- `encoder_width` and `log_std_init`

These are **never re-tuned** in this study; only reddit parameters and
PPO/training hyperparameters are searched.


In [ ]:
def load_frozen_config(db_path: str, study_name: str) -> Dict[str, Any]:
    """
    Load the best trial from the v9.1.3 Optuna DB.
    Returns a dict with frozen technical feature picks and architecture params.
    """
    storage = f'sqlite:///{db_path}'
    study   = optuna.load_study(study_name=study_name, storage=storage)
    completed = [
        t for t in study.trials
        if t.state == TrialState.COMPLETE
        and t.value is not None
        and np.isfinite(t.value)
    ]
    if not completed:
        raise ValueError('No completed trials found in v9.1.3 DB.')
    best = max(completed, key=lambda t: t.value)

    feature_picks    = best.user_attrs.get('feature_set', {})
    per_asset_feats  = best.user_attrs.get('per_asset_feats', [])
    lookback         = best.user_attrs.get('lookback', best.params.get('lookback', 22))
    encoder_width    = best.params.get('encoder_width', 64)
    log_std_init     = best.params.get('log_std_init', -1.0)
    total_timesteps  = best.user_attrs.get('total_timesteps', 1_000_000)

    print(f'Loaded frozen config from v9.1.3:')
    print(f'  Trial #{best.number}  val_sharpe={best.value:.4f}')
    print(f'  feature_picks:   {feature_picks}')
    print(f'  per_asset_feats: {per_asset_feats}')
    print(f'  lookback={lookback}  encoder_width={encoder_width}  '
          f'log_std_init={log_std_init}  total_timesteps={total_timesteps}')

    return {
        'feature_picks':   feature_picks,
        'per_asset_feats': per_asset_feats,
        'lookback':        int(lookback),
        'encoder_width':   int(encoder_width),
        'log_std_init':    float(log_std_init),
        'total_timesteps': int(total_timesteps),
        'trial_number':    best.number,
        'value':           best.value,
    }


# ── Run at notebook startup ───────────────────────────────────────────────
FROZEN_CONFIG = load_frozen_config(V913_DB_PATH, V913_STUDY_NAME)


---
## 7 · Reddit Feature Helpers

In [ ]:
def _group_to_key(g) -> str:
    """Serialise a group (str or tuple) as a stable Optuna categorical key."""
    return '+'.join(g) if isinstance(g, tuple) else str(g)


def _key_to_indicators(key: str) -> List[str]:
    """Expand an Optuna key back to a flat list of indicator names."""
    return [] if key == 'none' else key.split('+')


def pick_reddit_features(
    trial: optuna.Trial,
    live_groups: list = REDDIT_LIVE_GROUPS,
    hist_groups: list = REDDIT_HIST_GROUPS,
    thresholds: List[str] = REDDIT_THRESHOLDS,
) -> Dict[str, Any]:
    """
    Sample reddit configuration from Optuna.

    REDDIT_LIVE  — mandatory, exactly one group.
    REDDIT_HIST  — optional, 'none' = skip.
    Threshold  — shared across all active reddit features.
    Paired indicators always enter together.

    Returns dict with:
      'threshold'       : str
      'live_key'        : str
      'hist_key'        : str
      'live_indicators' : List[str]
      'hist_indicators' : List[str]  (empty when hist_key == 'none')
      'reddit_cols'  : List[str]  column suffixes, e.g.
                          ['reddit_threshold0.50_balance_ratio',
                           'reddit_threshold0.50_balance_ratio_t-1',
                           'reddit_threshold0.50_balance_ratio_t-5']
    """
    threshold = trial.suggest_categorical('reddit_threshold', thresholds)

    live_keys       = [_group_to_key(g) for g in live_groups]
    live_key        = trial.suggest_categorical('reddit_live_group', live_keys)
    live_indicators = _key_to_indicators(live_key)

    hist_keys       = [_group_to_key(g) for g in hist_groups]
    hist_key        = trial.suggest_categorical('reddit_hist_group', hist_keys)
    hist_indicators = _key_to_indicators(hist_key)

    prefix = f'reddit_threshold{threshold}'
    reddit_cols = (
        [f'{prefix}_{ind}' for ind in live_indicators]
        + [f'{prefix}_{ind}' for ind in hist_indicators]
    )
    return {
        'threshold':       threshold,
        'live_key':        live_key,
        'hist_key':        hist_key,
        'live_indicators': live_indicators,
        'hist_indicators': hist_indicators,
        'reddit_cols':  reddit_cols,
    }


def apply_reddit_nan_strategy(
    X_sent: np.ndarray,         # [T, S]  raw sentiment values
    reddit_nan_flag: np.ndarray,  # [T]     1 = feed down
    strategy: str,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Clean sentiment values on days where the feed was down.

    Returns:
      X_out      : [T, S]  cleaned values
      loss_mask  : [T] bool  True = exclude from reward (only for 'mask_asset')
    """
    missing   = reddit_nan_flag.astype(bool)
    X_out     = X_sent.copy().astype(np.float32)
    loss_mask = np.zeros(len(reddit_nan_flag), dtype=bool)

    if strategy == 'zero_fill':
        X_out[missing] = 0.0
    elif strategy == 'carry_forward':
        X_out[missing] = np.nan
        df_tmp = pd.DataFrame(X_out).ffill().fillna(0.0)
        X_out  = df_tmp.to_numpy(dtype=np.float32)
    elif strategy == 'mask_asset':
        X_out[missing] = 0.0
        loss_mask       = missing
    else:
        raise ValueError(f'Unknown reddit_nan_strategy: {strategy!r}')

    return X_out, loss_mask


---
## 8 · Feature Engineering Helpers  *(extended for sentiment)*

In [ ]:
def get_per_asset_feature_list(picks: Dict[str, Any]) -> List[str]:
    """Ordered, deduplicated list of per-asset TECHNICAL feature names."""
    feats: List[str] = ['log_ret_1d']
    for f in picks.get('OHLC', []):
        feats.append(f)
    for bucket in ['Trend', 'Momentum', 'Volume', 'Volatility']:
        val = picks[bucket]
        if isinstance(val, list):
            feats.extend(val)
        else:
            feats.append(val)
    seen: set = set()
    out: List[str] = []
    for f in feats:
        if f not in seen:
            out.append(f)
            seen.add(f)
    return out


def required_df_columns(
    assets: List[str],
    per_asset_feats: List[str],
    picks: Dict[str, Any],
    reddit_cols: Optional[List[str]] = None,  # suffixes only
) -> List[str]:
    cols: List[str] = []
    for a in assets:
        cols.append(f'{a}_Close')
        for f in per_asset_feats:
            if f == 'log_ret_1d':
                continue
            cols.append(f'{a}_{f}')
        if reddit_cols:
            for sf in reddit_cols:
                cols.append(f'{a}_{sf}')
    if reddit_cols:
        cols.append('_REDDIT_NAN')   # single global column — appended once
    if picks.get('Market') is not None:
        cols.append(picks['Market'])
    return list(dict.fromkeys(cols))


---
## 9 · Stationary Feature Transforms  *(extended for sentiment + `_REDDIT_NAN`)*

All transform parameters are fit on the training window only.

| Feature type | Transform |
|---|---|
| Technical features | same as v9.1.3 |
| Reddit indicators (per-asset) | robust z-score |
| `_REDDIT_NAN` flag | identity (binary 0/1, no scaling) |


In [ ]:
def _winsorize_fit(x: np.ndarray, q: float) -> Tuple[float, float]:
    lo = np.nanquantile(x, q)
    hi = np.nanquantile(x, 1.0 - q)
    if not np.isfinite(lo): lo = 0.0
    if not np.isfinite(hi): hi = 0.0
    if hi < lo: hi = lo
    return float(lo), float(hi)

def _winsorize_apply(x: np.ndarray, lo: float, hi: float) -> np.ndarray:
    return np.clip(x, lo, hi)


@dataclass
class FeatureTransformSpec:
    kind:       str    # 'robust_z' | 'bounded_0_100' | 'log1p_robust_z' | 'identity' | 'clip'
    winsor_q:   float = 0.01
    clip_value: float = 10.0


@dataclass
class FittedTransform:
    spec:   FeatureTransformSpec
    lo:     float = 0.0
    hi:     float = 0.0
    center: float = 0.0
    scale:  float = 1.0

    def transform(self, x: np.ndarray) -> np.ndarray:
        x = x.astype(np.float32)
        x = np.nan_to_num(x, nan=np.nan, posinf=np.nan, neginf=np.nan)

        if self.spec.kind in ('robust_z', 'log1p_robust_z'):
            x = _winsorize_apply(x, self.lo, self.hi)

        if self.spec.kind == 'identity':
            y = x
        elif self.spec.kind == 'bounded_0_100':
            y = (x - 50.0) / 50.0
        elif self.spec.kind == 'log1p_robust_z':
            y = np.sign(x) * np.log1p(np.abs(x))
            y = (y - self.center) / self.scale
        elif self.spec.kind == 'robust_z':
            y = (x - self.center) / self.scale
        elif self.spec.kind == 'clip':
            y = np.clip(x, -self.spec.clip_value, self.spec.clip_value)
        else:
            raise ValueError(f'Unknown transform kind: {self.spec.kind}')

        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
        if self.spec.clip_value is not None:
            y = np.clip(y, -self.spec.clip_value, self.spec.clip_value)
        return y.astype(np.float32)


def default_transform_spec(feature_name: str) -> FeatureTransformSpec:
    """Choose the appropriate transformation by feature semantics.
    _REDDIT_NAN  → identity (binary flag, no scaling).
    Reddit indicators → robust_z (continuous, potentially skewed).
    Technical indicators → same as v9.1.3.
    """
    if feature_name == '_REDDIT_NAN':
        return FeatureTransformSpec(kind='identity', clip_value=2.0)
    # Sentiment indicator columns contain 'reddit_threshold' in their name
    if 'reddit_threshold' in feature_name:
        return FeatureTransformSpec(kind='robust_z', winsor_q=0.01, clip_value=8.0)
    # Technical features (unchanged from v9.1.3)
    name = feature_name.lower()
    if feature_name == 'log_ret_1d':
        return FeatureTransformSpec(kind='robust_z', winsor_q=0.01, clip_value=8.0)
    if name in ('rsi', 'adx'):
        return FeatureTransformSpec(kind='bounded_0_100', clip_value=2.0)
    if name in ('volume', 'volumevariation', 'obv', 'adi'):
        return FeatureTransformSpec(kind='log1p_robust_z', winsor_q=0.01, clip_value=8.0)
    if name in ('open', 'high', 'low'):
        return FeatureTransformSpec(kind='robust_z', winsor_q=0.01, clip_value=8.0)
    return FeatureTransformSpec(kind='robust_z', winsor_q=0.01, clip_value=8.0)


def fit_transforms_on_train(
    X_train_raw:      np.ndarray,            # [T, N, F]  (F includes sentiment + _REDDIT_NAN)
    feat_names:       List[str],
    market_train_raw: Optional[np.ndarray],  # [T, 1] or None
) -> Tuple[List[FittedTransform], Optional[FittedTransform]]:
    """Fit per-feature transforms by pooling over time and assets.
    _REDDIT_NAN gets identity; sentiment columns get robust_z.
    """
    fitted: List[FittedTransform] = []
    for j, fname in enumerate(feat_names):
        spec = default_transform_spec(fname)
        if spec.kind in ('identity', 'bounded_0_100', 'clip'):
            fitted.append(FittedTransform(spec=spec))
            continue
        vals = X_train_raw[:, :, j].reshape(-1)
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            fitted.append(FittedTransform(spec=spec))
            continue
        if spec.kind in ('robust_z', 'log1p_robust_z'):
            lo, hi    = _winsorize_fit(vals, spec.winsor_q)
            vals_w    = _winsorize_apply(vals, lo, hi)
            if spec.kind == 'log1p_robust_z':
                vals_w = np.sign(vals_w) * np.log1p(np.abs(vals_w))
            center = float(np.nanmedian(vals_w))
            mad    = float(np.nanmedian(np.abs(vals_w - center)))
            scale  = 1.4826 * mad
            if not np.isfinite(scale) or scale < 1e-6:
                scale = float(np.nanstd(vals_w))
            if not np.isfinite(scale) or scale < 1e-6:
                scale = 1.0
            fitted.append(FittedTransform(spec=spec, lo=lo, hi=hi,
                                          center=center, scale=scale))
        else:
            raise ValueError(spec.kind)

    market_ft: Optional[FittedTransform] = None
    if market_train_raw is not None:
        spec = FeatureTransformSpec(kind='robust_z', winsor_q=0.01, clip_value=8.0)
        v    = market_train_raw[:, 0]
        v    = v[np.isfinite(v)]
        lr   = np.log(v[1:] / v[:-1]) if v.size > 2 else np.array([0.0], dtype=np.float32)
        lr   = lr[np.isfinite(lr)]
        lo, hi    = _winsorize_fit(lr, spec.winsor_q)
        lr_w      = _winsorize_apply(lr, lo, hi)
        center    = float(np.nanmedian(lr_w))
        mad       = float(np.nanmedian(np.abs(lr_w - center)))
        scale     = 1.4826 * mad
        if not np.isfinite(scale) or scale < 1e-6:
            scale = float(np.nanstd(lr_w))
        if not np.isfinite(scale) or scale < 1e-6:
            scale = 1.0
        market_ft = FittedTransform(spec=spec, lo=lo, hi=hi, center=center, scale=scale)

    return fitted, market_ft


def apply_transforms(
    X_raw:      np.ndarray,
    fitted:     List[FittedTransform],
    market_raw: Optional[np.ndarray],
    market_ft:  Optional[FittedTransform],
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    X = np.empty_like(X_raw, dtype=np.float32)
    for j, ft in enumerate(fitted):
        X[:, :, j] = ft.transform(X_raw[:, :, j])

    m_out: Optional[np.ndarray] = None
    if market_raw is not None and market_ft is not None:
        v    = market_raw[:, 0].astype(np.float32)
        lr   = np.full_like(v, np.nan, dtype=np.float32)
        lr[1:] = np.log(v[1:] / v[:-1])
        lr   = np.nan_to_num(lr, nan=0.0, posinf=0.0, neginf=0.0)
        m_out = market_ft.transform(lr).reshape(-1, 1).astype(np.float32)

    return X, m_out


---
## 10 · Build Raw Arrays  *(extended for sentiment + `_REDDIT_NAN`)*

`build_raw_arrays_with_reddit` is the replacement for v9.1.3's `build_raw_arrays`.

Array layout of `X_raw` (dim 2):
```
[0 .. F-1]          technical features (same as v9.1.3)
[F .. F+S-1]        per-asset reddit indicator values (S = len(reddit_cols))
[F+S]               _REDDIT_NAN flag (same scalar broadcast to all N assets)
```

Total feature count = F + R + 1.
The availability mask is computed from technical features only; the
`_REDDIT_NAN` flag is never treated as a finite-feature requirement.


In [ ]:
def build_raw_arrays_with_reddit(
    df:                pd.DataFrame,
    assets:            List[str],
    picks:             Dict[str, Any],
    logger:            logging.Logger,
    reddit_cols:    List[str],      # suffixes, e.g. ['reddit_threshold0.50_balance_ratio']
    reddit_nan_strategy: str,
) -> Tuple[np.ndarray, np.ndarray, List[str], Optional[np.ndarray]]:
    """
    Extended build_raw_arrays that appends reddit features and the _REDDIT_NAN flag.

    _REDDIT_NAN is a SINGLE GLOBAL column — read once, broadcast to all N assets.
    The NaN strategy is applied globally: on feed-down days, ALL assets are affected.

    Returns:
      closes     : [T, N]     raw close prices
      X_raw      : [T, N, F+S+1]  base feats + sentiment feats + _REDDIT_NAN
      feat_names : List[str]  length = F + R + 1
      market     : [T, 1] raw VIX close, or None
    """
    per_asset_feats = get_per_asset_feature_list(picks)
    cols = required_df_columns(assets, per_asset_feats, picks, reddit_cols)
    missing = [c for c in cols if c not in df.columns]
    if missing:
        logger.error(f'Missing columns ({len(missing)}): {missing[:25]}')
        raise KeyError('Dataset missing required columns.')

    T = len(df)
    N = len(assets)
    F = len(per_asset_feats)
    R = len(reddit_cols)
    F_total = F + R + 1   # +1 for _REDDIT_NAN

    closes = np.zeros((T, N), dtype=np.float32)
    X_raw  = np.zeros((T, N, F_total), dtype=np.float32)

    # ── Load global _REDDIT_NAN flag ONCE ───────────────────────────────────
    reddit_nan_flag = df['_REDDIT_NAN'].to_numpy(np.float32)   # [T], 1 = feed down

    for i, a in enumerate(assets):
        c = df[f'{a}_Close'].to_numpy(np.float32)
        closes[:, i] = c

        # ── Technical features (identical to v9.1.3 build_raw_arrays) ─────
        for j, f in enumerate(per_asset_feats):
            if f == 'log_ret_1d':
                lr    = np.full_like(c, np.nan, dtype=np.float32)
                lr[1:] = np.log(c[1:] / c[:-1])
                X_raw[:, i, j] = lr
            elif f in ('Open', 'High', 'Low'):
                raw   = df[f'{a}_{f}'].to_numpy(np.float32)
                rel   = np.full_like(raw, np.nan, dtype=np.float32)
                valid = (raw > 0) & (c > 0) & np.isfinite(raw) & np.isfinite(c)
                rel[valid] = np.log(raw[valid] / c[valid])
                X_raw[:, i, j] = rel
            elif f.startswith('SMA'):
                sma   = df[f'{a}_{f}'].to_numpy(np.float32)
                rel   = np.full_like(sma, np.nan, dtype=np.float32)
                valid = (sma > 0) & (c > 0) & np.isfinite(sma) & np.isfinite(c)
                rel[valid] = np.log(c[valid] / sma[valid])
                X_raw[:, i, j] = rel
            else:
                X_raw[:, i, j] = df[f'{a}_{f}'].to_numpy(np.float32)

        # ── Reddit features (per-asset values, global missingness) ──────
        if R > 0:
            sent_raw = np.stack(
                [df[f'{a}_{sf}'].to_numpy(np.float32) for sf in reddit_cols],
                axis=1,
            )  # [T, R]
            sent_clean, _ = apply_reddit_nan_strategy(
                sent_raw, reddit_nan_flag, reddit_nan_strategy)
            X_raw[:, i, F:F + R] = sent_clean

        # ── _REDDIT_NAN flag — broadcast same global flag to every asset ─────
        X_raw[:, i, F + R] = reddit_nan_flag

    market: Optional[np.ndarray] = None
    if picks.get('Market') is not None:
        market = df[picks['Market']].to_numpy(np.float32).reshape(-1, 1)

    feat_names = list(per_asset_feats) + list(reddit_cols) + ['_REDDIT_NAN']
    return closes, X_raw, feat_names, market


---
## 11 · Availability Mask

Same logic as v9.1.3.  
The mask is computed only from the **technical feature** slice (dims `[0..F-1]`).
Sentiment and `_REDDIT_NAN` dims are excluded — `_REDDIT_NAN` is always 0 or 1
and should not drive availability.


In [ ]:
def compute_availability_mask(
    closes_raw:    np.ndarray,   # [T, N]
    feats_raw:     np.ndarray,   # [T, N, F+R+1]  full array
    lookback:      int,
    n_tech_feats:  int,          # F  (number of technical features — slice [:n_tech_feats])
) -> np.ndarray:
    """
    Availability mask based on technical features only.
    `_REDDIT_NAN` and sentiment dims are excluded so feed-down days
    are not forced-unavailable — the NaN strategy handles them.
    """
    T, N = closes_raw.shape
    avail = np.zeros((T, N), dtype=bool)

    finite_close_t  = np.isfinite(closes_raw)
    finite_close_t1 = np.isfinite(
        np.vstack([closes_raw[1:], np.full((1, N), np.nan, dtype=np.float32)])
    )
    # Only check the technical feature slice
    finite_feats_tech = np.isfinite(feats_raw[:, :, :n_tech_feats]).all(axis=2)  # [T, N]

    for t in range(lookback - 1, T - 1):
        win_ok = finite_feats_tech[t - lookback + 1: t + 1].all(axis=0)
        avail[t] = finite_close_t[t] & finite_close_t1[t] & win_ok

    avail[-1, :] = False
    return avail


---
## 12 · Differential Sharpe Ratio (DSR) Reward  *(unchanged)*

In [ ]:
@dataclass
class DiffSharpeState:
    A:    float = 0.0
    B:    float = 0.0
    init: bool  = True


def diff_sharpe_update(
    state: DiffSharpeState, r: float, eta: float = 0.01, eps: float = 1e-8
) -> float:
    if state.init:
        state.A    = r
        state.B    = r * r
        state.init = False
        return 0.0
    A_prev   = state.A
    B_prev   = state.B
    var_prev = max(B_prev - A_prev * A_prev, 0.0)
    num = (B_prev * (r - A_prev)) - 0.5 * A_prev * ((r * r) - B_prev)
    den = (var_prev ** 1.5) + eps
    dsr = float(num / den)
    state.A = A_prev + eta * (r - A_prev)
    state.B = B_prev + eta * ((r * r) - B_prev)
    return 0.0 if not np.isfinite(dsr) else dsr


---
## 13 · Portfolio Environment  *(unchanged from v9.1.3)*

In [ ]:
class OneDayHoldPortfolioEnv(gym.Env):
    metadata = {'render_modes': []}

    def __init__(
        self,
        closes_raw:       np.ndarray,
        feats:            np.ndarray,
        avail_mask:       np.ndarray,
        spreads:          np.ndarray,
        market:           Optional[np.ndarray] = None,
        dsr_eta:          float = 0.01,
        dsr_clip:         float = 10.0,
        dsr_warmup_steps: int   = 252,
        reward_scale_net: float = 100.0,
        reward_scale_dsr: float = 10.0,
        start_idx:        int   = 0,
        end_idx:          Optional[int] = None,
        lookback:         int   = 22,
    ):
        super().__init__()
        self.closes  = closes_raw.astype(np.float32)
        self.feats   = feats.astype(np.float32)
        self.avail   = avail_mask
        self.market  = market.astype(np.float32) if market is not None else None
        self.spreads = spreads.astype(np.float32)

        self.dsr_eta          = float(dsr_eta)
        self.dsr_clip         = float(dsr_clip)
        self.dsr_warmup_steps = int(dsr_warmup_steps)
        self.reward_scale_net = float(reward_scale_net)
        self.reward_scale_dsr = float(reward_scale_dsr)

        self.N         = self.closes.shape[1]
        self.F         = self.feats.shape[2]
        self.T         = self.closes.shape[0]
        self.start_idx = int(start_idx)
        self.end_idx   = int(end_idx) if end_idx is not None else (self.T - 2)
        self.lookback  = int(lookback)

        self.action_space = spaces.Box(low=-5.0, high=5.0, shape=(self.N,), dtype=np.float32)
        obs_dim = self.lookback * self.N * self.F + self.N + self.N
        if self.market is not None:
            obs_dim += 1
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)

        self._t = self._w_prev = self._dsr = self._step = None

    @staticmethod
    def _masked_softmax(a: np.ndarray, mask: np.ndarray) -> np.ndarray:
        a = a.astype(np.float32)
        if not mask.any():
            return np.zeros_like(a, dtype=np.float32)
        logits = a.copy(); logits[~mask] = -1e9
        logits -= np.max(logits[mask])
        e = np.zeros_like(logits, dtype=np.float32)
        e[mask] = np.exp(logits[mask])
        s = float(np.sum(e[mask]))
        w = np.zeros_like(e, dtype=np.float32)
        w[mask] = e[mask] / max(s, 1e-12)
        return w

    @staticmethod
    def _entropy(w: np.ndarray) -> float:
        w = np.asarray(w, dtype=np.float64)
        return float(-np.sum(w * np.log(w + 1e-12)))

    def _get_obs(self) -> np.ndarray:
        x = self.feats[self._t - self.lookback + 1: self._t + 1].reshape(-1)
        parts = [x]
        if self.market is not None:
            parts.append(self.market[self._t].reshape(-1))
        parts.append(self._w_prev.astype(np.float32))
        parts.append(self.avail[self._t].astype(np.float32))
        return np.nan_to_num(
            np.concatenate(parts).astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

    def reset(self, *, seed: Optional[int] = None, options: Optional[dict] = None):
        super().reset(seed=seed)
        self._t      = max(self.start_idx, self.lookback - 1)
        self._w_prev = np.ones(self.N, dtype=np.float32) / self.N
        self._dsr    = DiffSharpeState()
        self._step   = 0
        return self._get_obs(), {}

    def step(self, action: np.ndarray):
        t = self._t
        if t >= self.end_idx:
            w  = self._w_prev if self._w_prev is not None else np.ones(self.N) / self.N
            ew = 1.0 / self.N
            info = {
                'net_ret': 0.0, 'gross_ret': 0.0, 'cost': 0.0, 'turnover': 0.0,
                'dsr': 0.0, 'w': w.astype(np.float32),
                'l1_to_ew': float(np.sum(np.abs(w - ew))),
                'went': self._entropy(w), 'wmax': float(np.max(w)),
                'tradable_n': int(self.avail[t].sum()) if t < self.avail.shape[0] else 0,
            }
            return self._get_obs(), 0.0, True, False, info

        mask  = self.avail[t]
        w_new = self._masked_softmax(action, mask)

        p0  = np.where(self.closes[t] <= 0, np.nan, self.closes[t])
        rel = np.where(
            np.isfinite(self.closes[t + 1] / p0 - 1.0),
            self.closes[t + 1] / p0 - 1.0, 0.0).astype(np.float32)

        gross    = float(np.dot(w_new, rel))
        dw       = w_new - self._w_prev
        turnover = float(np.sum(np.abs(dw)))
        cost     = float(0.5 * np.sum(np.abs(dw) * self.spreads))
        net      = gross - cost

        dsr_val = float(np.clip(
            diff_sharpe_update(self._dsr, net, eta=self.dsr_eta),
            -self.dsr_clip, self.dsr_clip))

        if self.dsr_warmup_steps > 0 and self._step < self.dsr_warmup_steps:
            alpha  = self._step / self.dsr_warmup_steps
            reward = ((1.0 - alpha) * self.reward_scale_net * net
                      + alpha       * self.reward_scale_dsr * dsr_val)
        else:
            reward = self.reward_scale_dsr * dsr_val

        self._step += 1; self._w_prev = w_new; self._t += 1

        ew   = 1.0 / self.N
        info = {
            'net_ret': net, 'gross_ret': gross, 'cost': cost, 'turnover': turnover,
            'dsr': dsr_val, 'w': w_new.astype(np.float32),
            'l1_to_ew': float(np.sum(np.abs(w_new - ew))),
            'went': self._entropy(w_new), 'wmax': float(np.max(w_new)),
            'tradable_n': int(mask.sum()),
        }
        return self._get_obs(), float(reward), False, False, info


---
## 14 · Metrics & Plotting  *(unchanged)*

In [ ]:
def annualized_sharpe(returns: np.ndarray, eps: float = 1e-12, periods_per_year: int = 252) -> float:
    r = np.asarray(returns, dtype=np.float64)
    if r.size < 10: return float('nan')
    mu = float(np.mean(r)); sd = float(np.std(r))
    if sd < eps: return float('nan')
    return float((mu / sd) * math.sqrt(periods_per_year))

def annualized_return(returns: np.ndarray, periods_per_year: int = 252) -> float:
    r = np.asarray(returns, dtype=np.float64)
    if r.size == 0: return float('nan')
    wealth = np.prod(1.0 + r); years = len(r) / periods_per_year
    if years <= 0 or wealth <= 0: return float('nan')
    return float(wealth ** (1.0 / years) - 1.0)

def max_drawdown(returns: np.ndarray) -> float:
    wealth = _cum_wealth(returns)
    peak   = np.maximum.accumulate(wealth)
    return float(np.min(wealth / peak - 1.0))

def _cum_wealth(rets: np.ndarray) -> np.ndarray:
    return np.cumprod(1.0 + np.asarray(rets, dtype=np.float64))

def rollout(
    model: PPO, env: gym.Env,
    deterministic: bool = True,
    dates_index: Optional[pd.DatetimeIndex] = None,
) -> Dict[str, Any]:
    obs, _ = env.reset(); done = False
    nets, grosses, costs, turns, ws, l1s, ents, wmaxs, tradables = ([] for _ in range(9))
    step_ts: List[int] = []
    while not done:
        step_t = int(env._t)
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, _, done, trunc, info = env.step(action)
        nets.append(info['net_ret']); grosses.append(info['gross_ret'])
        costs.append(info['cost']);   turns.append(info['turnover'])
        ws.append(info['w']);         l1s.append(info['l1_to_ew'])
        step_ts.append(step_t)
        ents.append(info['went']);    wmaxs.append(info['wmax'])
        tradables.append(info['tradable_n'])
        if trunc: break
    nets = np.asarray(nets, dtype=np.float32); ws = np.asarray(ws, dtype=np.float32)
    dates_out = (dates_index[step_ts]
                 if dates_index is not None and len(step_ts) > 0 else None)
    return {
        'net': nets, 'gross': np.asarray(grosses, dtype=np.float32), 'w': ws,
        'sharpe': annualized_sharpe(nets), 'ann_return': annualized_return(nets),
        'max_drawdown': max_drawdown(nets),
        'mu': float(nets.mean()), 'sd': float(nets.std()),
        'cost_mu': float(np.mean(costs)), 'turnover_mu': float(np.mean(turns)),
        'l1_mu': float(np.mean(l1s)), 'ent_mu': float(np.mean(ents)),
        'wmax_mu': float(np.mean(wmaxs)), 'tradable_n_mu': float(np.mean(tradables)),
        'n': int(len(nets)),
        'dates': dates_out,
    }

def rollout_equal_weight(env: 'OneDayHoldPortfolioEnv') -> Dict[str, Any]:
    obs, _ = env.reset(); done = False; nets, grosses = [], []
    action = np.zeros(env.N, dtype=np.float32)
    while not done:
        obs, _, done, trunc, info = env.step(action)
        nets.append(info['net_ret']); grosses.append(info['gross_ret'])
        if trunc: break
    nets = np.asarray(nets, dtype=np.float32)
    return {
        'net': nets, 'gross': np.asarray(grosses, dtype=np.float32),
        'sharpe': annualized_sharpe(nets), 'ann_return': annualized_return(nets),
        'max_drawdown': max_drawdown(nets), 'mu': float(nets.mean()),
        'sd': float(nets.std()), 'n': int(len(nets)),
    }

def save_plots(out_dir, prefix, agent, ew, asset_names):
    os.makedirs(out_dir, exist_ok=True)
    plt.figure()
    plt.plot(_cum_wealth(agent['net']), label='agent')
    plt.plot(_cum_wealth(ew['net']),    label='equal_weight')
    plt.title(f'{prefix} cumulative wealth')
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'{prefix}_cum.png')); plt.close()
    if 'w' in agent and agent['w'].ndim == 2 and agent['w'].shape[0] > 5:
        plt.figure()
        plt.stackplot(np.arange(agent['w'].shape[0]), agent['w'].T, labels=asset_names)
        plt.title(f'{prefix} weights'); plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f'{prefix}_weights.png')); plt.close()

def save_metrics_json(path, metrics):
    out = {}
    for k, v in metrics.items():
        if isinstance(v, (np.ndarray, pd.DatetimeIndex)): continue
        out[k] = float(v) if isinstance(v, (np.floating, np.integer)) else v
    with open(path, 'w') as f: json.dump(out, f, indent=2)


---
## 15 · Training Callbacks  *(unchanged)*

In [ ]:
class EarlyStopSharpeCallback(BaseCallback):
    def __init__(self, eval_env_fn, eval_freq, patience_evals,
                 min_evals=5, min_delta=0.0, deterministic=True, verbose=0):
        super().__init__(verbose=verbose)
        self.eval_env_fn    = eval_env_fn
        self.eval_freq      = int(eval_freq)
        self.patience_evals = int(patience_evals)
        self.min_evals      = int(min_evals)
        self.min_delta      = float(min_delta)
        self.deterministic  = deterministic
        self.best_sharpe         = -np.inf
        self.best_num_timesteps  = 0
        self.no_improve_count    = 0
        self.eval_count          = 0
        self.best_state_dict     = None

    def _on_step(self) -> bool:
        if self.num_timesteps % self.eval_freq != 0:
            return True
        self.eval_count += 1
        sh = float(rollout(self.model, self.eval_env_fn(),
                           deterministic=self.deterministic)['sharpe'])
        self.logger.record('early_stop/es_sharpe', sh)
        self.logger.dump(self.num_timesteps)
        if np.isfinite(sh) and sh > self.best_sharpe + self.min_delta:
            self.best_sharpe        = sh
            self.best_num_timesteps = self.num_timesteps
            self.no_improve_count   = 0
            self.best_state_dict    = {k: v.detach().cpu().clone()
                                       for k, v in self.model.policy.state_dict().items()}
        else:
            self.no_improve_count += 1
        if self.eval_count >= self.min_evals and self.no_improve_count >= self.patience_evals:
            return False
        return True

    def restore_best_model(self):
        if self.best_state_dict is not None:
            self.model.policy.load_state_dict(self.best_state_dict)


class DiagnosticsCallback(BaseCallback):
    def __init__(self, eval_every_steps, train_env_fn, es_val_env_fn,
                 test_env_fn, name, deterministic=True, verbose=0):
        super().__init__(verbose=verbose)
        self.eval_every_steps = int(eval_every_steps)
        self.train_env_fn     = train_env_fn
        self.es_val_env_fn    = es_val_env_fn
        self.test_env_fn      = test_env_fn
        self.name             = name
        self.deterministic    = deterministic

    def _on_step(self) -> bool:
        if self.num_timesteps % self.eval_every_steps != 0:
            return True
        tr = rollout(self.model, self.train_env_fn(),  deterministic=self.deterministic)
        es = rollout(self.model, self.es_val_env_fn(), deterministic=self.deterministic)
        te = rollout(self.model, self.test_env_fn(),   deterministic=self.deterministic)
        print(f'[{self.name}] step={self.num_timesteps} '
              f"TR sh={tr['sharpe']:.3f} | ES sh={es['sharpe']:.3f} | TEST sh={te['sharpe']:.3f}")
        self.logger.record('diag/train_sharpe',  tr['sharpe'])
        self.logger.record('diag/es_val_sharpe', es['sharpe'])
        self.logger.record('diag/test_sharpe',   te['sharpe'])
        self.logger.dump(self.num_timesteps)
        return True


---
## 16 · Low-Level Trainer  *(unchanged)*

In [ ]:
def make_vec_env(env_ctor):
    return DummyVecEnv([env_ctor])


def _build_and_train_ppo(
    vec_train,
    ppo_params:      Dict[str, Any],
    total_timesteps: int,
    es_callback:     EarlyStopSharpeCallback,
    diag_callback:   DiagnosticsCallback,
    seed:            int,
    tb_path:         str,
) -> PPO:
    np.random.seed(seed); random.seed(seed); th.manual_seed(seed)

    model = PPO(
        'MlpPolicy', vec_train, seed=seed, verbose=0,
        policy_kwargs=ppo_params['policy_kwargs'],
        gamma=ppo_params['gamma'], gae_lambda=ppo_params['gae_lambda'],
        learning_rate=ppo_params['learning_rate'], clip_range=ppo_params['clip_range'],
        n_epochs=ppo_params['n_epochs'], n_steps=ppo_params['n_steps'],
        batch_size=ppo_params['batch_size'], ent_coef=ppo_params['ent_coef'],
        vf_coef=ppo_params['vf_coef'], max_grad_norm=ppo_params['max_grad_norm'],
        target_kl=ppo_params['target_kl'],
    )
    model.learn(
        total_timesteps=total_timesteps,
        callback=CallbackList([es_callback, diag_callback]),
        progress_bar=False,
    )
    es_callback.restore_best_model()
    return model


---
## 17 · Search-Space Size Estimator  *(unchanged)*

In [ ]:
def estimate_search_space(study: optuna.Study) -> int:
    if not study.trials:
        raise ValueError('Run at least 1 trial first so distributions are populated.')
    distributions         = study.trials[0].distributions
    discrete_combinations = 1
    continuous_params: List[str] = []
    print('Search space:')
    for name, dist in distributions.items():
        if isinstance(dist, optuna.distributions.CategoricalDistribution):
            n = len(dist.choices)
            discrete_combinations *= n
            print(f'  {name:35s}  categorical  {n:>4d} choices')
        elif isinstance(dist, optuna.distributions.IntDistribution):
            n = (dist.high - dist.low) // dist.step + 1
            discrete_combinations *= n
            print(f'  {name:35s}  int          {n:>4d} levels  [{dist.low}..{dist.high}]')
        elif isinstance(dist, optuna.distributions.FloatDistribution):
            continuous_params.append(name)
            scale = 'log' if dist.log else 'linear'
            print(f'  {name:35s}  float ({scale:6s})  [{dist.low:.2e}..{dist.high:.2e}]  ← continuous')
    suggested = math.ceil(math.sqrt(discrete_combinations))
    print(f'\nDiscrete combinations : {discrete_combinations}')
    print(f'Continuous params     : {continuous_params}')
    print(f'Suggested N_TRIALS    : ceil(sqrt({discrete_combinations})) = {suggested}')
    return suggested


---
## 18 · Optuna Objective & `run_optuna_search`  *(reddit version)*

Each trial:
1. Uses **frozen** technical feature picks from `FROZEN_CONFIG` (v9.1.3 best trial)
2. Samples reddit sentiment params (live group, hist group, threshold, NaN strategy)
3. Samples **all PPO / training hyperparameters** (same space as v9.1.3)
4. Trains 3 folds × N seeds → mean Sharpe (same objective as v9.1.3)

The `lookback` and `encoder_width` from the frozen config are re-used as
**starting values** but are still tunable — you can freeze them too by
removing them from the suggestion block below if desired.


In [ ]:
def reddit_ppo_objective_factory(
    df:           pd.DataFrame,
    assets:       List[str],
    spreads:      np.ndarray,
    frozen_config: Dict[str, Any],
    base_log_dir: str,
    seeds:        List[int],
):
    dates_index = pd.DatetimeIndex(df.index)
    splits      = make_walk_forward_splits(dates_index)
    picks       = frozen_config['feature_picks']   # frozen technical picks

    def objective(trial: optuna.Trial) -> float:
        trial_dir = os.path.join(base_log_dir, f'trial_{trial.number:05d}')
        logger    = setup_logger(trial_dir, name_prefix='ppo_reddit_tune')

        logger.info('=' * 80)
        logger.info(f'TRIAL {trial.number} | frozen tech picks: {picks}')

        # ── Reddit search space ────────────────────────────────────────
        sent_picks     = pick_reddit_features(trial)
        reddit_cols = sent_picks['reddit_cols']
        nan_strategy   = trial.suggest_categorical(
            'reddit_nan_strategy', ['zero_fill', 'carry_forward', 'mask_asset'])

        trial.set_user_attr('reddit_picks',   sent_picks)
        trial.set_user_attr('reddit_nan_strategy', nan_strategy)
        trial.set_user_attr('frozen_picks',      picks)
        logger.info(f'Sentiment: {sent_picks}  NaN strategy: {nan_strategy}')

        # ── PPO hyperparameters (same space as v9.1.3) ────────────────────
        gamma           = trial.suggest_float('gamma',         0.90, 0.999)
        gae_lambda      = trial.suggest_float('gae_lambda',    0.80, 0.98)
        learning_rate   = trial.suggest_float('learning_rate', 1e-5, 2e-3, log=True)
        clip_range      = trial.suggest_float('clip_range',    0.05, 0.20)
        n_epochs        = trial.suggest_int(  'n_epochs',      3, 10)

        n_steps     = trial.suggest_categorical('n_steps_v2',    [252, 504, 756])
        BATCH_SPACE = [63, 84, 108, 126, 168, 189, 252, 378, 504, 756]
        batch_size  = trial.suggest_categorical('batch_size_v2', BATCH_SPACE)

        valid_batch_sizes = {
            252: {63, 84, 126, 252},
            504: {84, 126, 168, 252, 504},
            756: {63, 84, 108, 126, 189, 252, 378, 756},
        }
        if batch_size not in valid_batch_sizes[n_steps] or batch_size > n_steps:
            raise optuna.exceptions.TrialPruned()

        target_kl     = trial.suggest_float('target_kl',     0.003, 0.03)
        ent_coef      = trial.suggest_float('ent_coef',      1e-5, 5e-2, log=True)
        vf_coef       = trial.suggest_float('vf_coef',       0.1, 1.0)
        max_grad_norm = trial.suggest_float('max_grad_norm', 0.3, 2.0)

        dsr_eta          = trial.suggest_categorical('dsr_eta',          [round(1/252, 6), 0.005, 0.01, 0.02])
        dsr_warmup_steps = trial.suggest_categorical('dsr_warmup_steps', [0, 126, 252, 504])
        reward_scale_net = trial.suggest_categorical('reward_scale_net', [50.0, 100.0, 200.0])
        reward_scale_dsr = trial.suggest_categorical('reward_scale_dsr', [5.0, 10.0, 20.0])
        dsr_clip         = trial.suggest_categorical('dsr_clip',         [5.0, 10.0, 20.0])

        # lookback and encoder_width: re-tune (can be frozen by hardcoding)
        lookback      = trial.suggest_categorical('lookback',      [5, 22, 60])
        encoder_width = trial.suggest_categorical('encoder_width', [32, 64, 128, 256])
        log_std_init  = trial.suggest_float('log_std_init', -1.5, 0.0)

        trial.set_user_attr('lookback', lookback)
        total_timesteps = frozen_config['total_timesteps']
        trial.set_user_attr('total_timesteps', total_timesteps)

        # ── Build raw arrays with sentiment ───────────────────────────────
        try:
            closes_raw, X_raw, feat_names, market_raw = build_raw_arrays_with_reddit(
                df=df, assets=assets, picks=picks,
                logger=logger, reddit_cols=reddit_cols,
                reddit_nan_strategy=nan_strategy,
            )
        except KeyError as e:
            logger.info(f'[PRUNE] Missing columns: {e}')
            raise optuna.TrialPruned()

        n_tech_feats = len(get_per_asset_feature_list(picks))
        avail = compute_availability_mask(closes_raw, X_raw, lookback, n_tech_feats)
        n_feats = X_raw.shape[2]   # F + R + 1

        has_market_flag = (picks.get('Market') is not None)
        policy_kwargs = dict(
            features_extractor_class=PerAssetFeaturesExtractor,
            features_extractor_kwargs=dict(
                n_assets=len(assets),
                lookback=lookback,
                n_features=n_feats,
                has_market=has_market_flag,
                encoder_width=encoder_width,
            ),
            net_arch=[encoder_width, encoder_width],
            activation_fn=nn.Tanh,
            log_std_init=log_std_init,
        )
        ppo_params = dict(
            gamma=gamma, gae_lambda=gae_lambda, learning_rate=learning_rate,
            clip_range=clip_range, n_epochs=n_epochs, n_steps=n_steps, batch_size=batch_size,
            target_kl=target_kl, ent_coef=ent_coef, vf_coef=vf_coef, max_grad_norm=max_grad_norm,
            policy_kwargs=policy_kwargs, dsr_warmup_steps=dsr_warmup_steps,
            reward_scale_net=reward_scale_net, reward_scale_dsr=reward_scale_dsr, dsr_clip=dsr_clip,
        )

        trial.set_user_attr('feat_names',  feat_names)
        trial.set_user_attr('has_market',  int(market_raw is not None))
        logger.info(f'feature_dim={n_feats} (tech={n_tech_feats}, '
                    f'reddit={n_feats - n_tech_feats - 1}, _REDDIT_NAN=1)')

        all_scores: List[float] = []

        for split_i, (train_idx, val_idx) in enumerate(splits):
            logger.info('-' * 80)
            logger.info(f'[SPLIT {split_i}] '
                        f'train {df.index[train_idx[0]]}..{df.index[train_idx[-1]]} '
                        f'val {df.index[val_idx[0]]}..{df.index[val_idx[-1]]}')

            X_train_raw      = X_raw[train_idx]
            market_train_raw = market_raw[train_idx] if market_raw is not None else None
            fitted, market_ft = fit_transforms_on_train(X_train_raw, feat_names, market_train_raw)
            X_trans, market_trans = apply_transforms(X_raw, fitted, market_raw, market_ft)

            if float(avail[val_idx].sum(axis=1).mean()) < 2.0:
                logger.info('[PRUNE] Too few tradable assets on validation.')
                raise optuna.exceptions.TrialPruned()

            seed_scores: List[float] = []
            for seed in seeds:
                train_core_idx, es_val_idx = split_train_for_early_stop(
                    train_idx, dates_index, es_years=1, min_train_days=252)

                tr_s = max(int(train_core_idx[0]),  lookback - 1)
                tr_e = min(int(train_core_idx[-1]), closes_raw.shape[0] - 2)
                es_s = max(int(es_val_idx[0]),      lookback - 1)
                es_e = min(int(es_val_idx[-1]),     closes_raw.shape[0] - 2)
                va_s = max(int(val_idx[0]),          lookback - 1)
                va_e = min(int(val_idx[-1]),         closes_raw.shape[0] - 2)

                def _env(s, e, Xt=X_trans, Mt=market_trans):
                    return OneDayHoldPortfolioEnv(
                        closes_raw=closes_raw, feats=Xt, market=Mt,
                        avail_mask=avail, spreads=spreads,
                        dsr_eta=dsr_eta, dsr_clip=dsr_clip,
                        dsr_warmup_steps=dsr_warmup_steps,
                        reward_scale_net=reward_scale_net,
                        reward_scale_dsr=reward_scale_dsr,
                        start_idx=s, end_idx=e, lookback=lookback,
                    )

                es_cb = EarlyStopSharpeCallback(
                    eval_env_fn=lambda s=es_s, e=es_e: _env(s, e),
                    eval_freq=50_000, patience_evals=4, min_evals=5, verbose=1)
                diag_cb = DiagnosticsCallback(
                    eval_every_steps=50_000,
                    train_env_fn  =lambda s=tr_s, e=tr_e: _env(s, e),
                    es_val_env_fn =lambda s=es_s, e=es_e: _env(s, e),
                    test_env_fn   =lambda s=va_s, e=va_e: _env(s, e),
                    name=f'split{split_i}_seed{seed}')

                tb_path = os.path.join(trial_dir, f'tb_split{split_i}_seed{seed}')
                model   = _build_and_train_ppo(
                    vec_train=make_vec_env(lambda s=tr_s, e=tr_e: _env(s, e)),
                    ppo_params=ppo_params, total_timesteps=total_timesteps,
                    es_callback=es_cb, diag_callback=diag_cb,
                    seed=seed, tb_path=tb_path,
                )

                sh = float(rollout(model, _env(va_s, va_e))['sharpe'])
                logger.info(f'[EVAL] split={split_i} seed={seed} val_sharpe={sh:.4f}')
                seed_scores.append(sh); all_scores.append(sh)

            logger.info(f'[SPLIT {split_i}] per_seed={seed_scores} '
                        f'mean={np.mean(seed_scores):.4f}')
            trial.report(float(np.mean(seed_scores)), step=split_i)
            if trial.should_prune():
                logger.info('[PRUNE] Optuna pruned this trial.')
                raise optuna.TrialPruned()

        score = float(np.mean(all_scores))
        trial.set_user_attr('all_scores', [float(x) for x in all_scores])
        logger.info('=' * 80)
        logger.info(f'TRIAL {trial.number} DONE  score={score:.4f}  all_scores={all_scores}')
        return score

    return objective


def run_optuna_search(
    df:            pd.DataFrame,
    assets:        List[str],
    spreads:       np.ndarray,
    frozen_config: Dict[str, Any],
    out_dir:       str,
    n_trials:      int = 50,
    seeds:         Optional[List[int]] = None,
) -> Tuple[optuna.Study, str]:
    os.makedirs(out_dir, exist_ok=True)
    seeds = seeds or [11, 22, 33]

    storage_path = os.path.join(out_dir, 'ppo_reddit_sentiment.db')
    storage      = f'sqlite:///{storage_path}'

    study = optuna.create_study(
        study_name='ppo_reddit_sentiment',
        direction='maximize',
        storage=storage,
        load_if_exists=True,
        sampler=TPESampler(seed=None, multivariate=True, group=True, constant_liar=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1),
    )

    objective = reddit_ppo_objective_factory(
        df=df, assets=assets, spreads=spreads,
        frozen_config=frozen_config, base_log_dir=out_dir, seeds=seeds,
    )
    study.optimize(objective, n_trials=n_trials, gc_after_trial=True, show_progress_bar=True)

    best = study.best_trial
    summary = {
        'best_value':      best.value,
        'best_params':     best.params,
        'best_user_attrs': {k: v for k, v in best.user_attrs.items()
                            if isinstance(v, (str, int, float, list, dict, bool))},
        'n_trials':        len(study.trials),
    }
    with open(os.path.join(out_dir, 'best_trial.json'), 'w') as f:
        json.dump(summary, f, indent=2)

    print('\n=== BEST TRIAL ===')
    print('Mean val Sharpe:', best.value)
    print('Sentiment:',       best.user_attrs.get('reddit_picks'))
    print('NaN strategy:',    best.user_attrs.get('reddit_nan_strategy'))
    print('DB:', storage_path)
    return study, out_dir


---
## 19 · Run Tuning

Set `N_TRIALS > 0` to start a real search. After the study completes, call
`estimate_search_space(study)` to gauge coverage, then proceed to Section 21.


In [ ]:
df = pd.read_parquet(DATASET_PATH, engine='pyarrow')

with open(SPREADS_PATH) as f:
    spread_dict = json.load(f)
spreads = np.array([spread_dict[a] for a in ASSETS], dtype=np.float32)

study, log_dir = run_optuna_search(
    df=df,
    assets=ASSETS,
    spreads=spreads,
    frozen_config=FROZEN_CONFIG,
    out_dir=OUT_DIR,
    n_trials=N_TRIALS,
    seeds=SEEDS,
)


In [ ]:
# ── Search-space diagnostics ──────────────────────────────────────────────
estimate_search_space(study)
print(f'You have ran {len(study.trials)} trials.')


---
## 20 · Load Best Trial

Reload the completed Optuna study and extract the best configuration.


In [ ]:
from pprint import pprint

storage_path = f'{OUT_DIR}/ppo_reddit_sentiment.db'
study = optuna.load_study(
    study_name='ppo_reddit_sentiment',
    storage=f'sqlite:///{storage_path}',
)

completed_trials = [
    t for t in study.trials
    if t.state == optuna.trial.TrialState.COMPLETE and t.value is not None
]
sorted_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)
best       = sorted_trials[0]
params     = best.params
user_attrs = best.user_attrs

# Frozen tech picks (from the v9.1.3 best trial)
feature_picks  = user_attrs['frozen_picks']

# Reddit config (tuned in this study)
reddit_picks  = user_attrs['reddit_picks']
reddit_nan_strategy = user_attrs['reddit_nan_strategy']

lookback         = user_attrs['lookback']
total_timesteps  = user_attrs.get('total_timesteps', FROZEN_CONFIG['total_timesteps'])
dsr_eta          = params['dsr_eta']
encoder_width    = params['encoder_width']
log_std_init     = params['log_std_init']

ppo_params = {
    'gamma':            params['gamma'],
    'gae_lambda':       params['gae_lambda'],
    'learning_rate':    params['learning_rate'],
    'clip_range':       params['clip_range'],
    'n_epochs':         params['n_epochs'],
    'n_steps':          params['n_steps_v2'],
    'batch_size':       params['batch_size_v2'],
    'target_kl':        params['target_kl'],
    'ent_coef':         params['ent_coef'],
    'vf_coef':          params['vf_coef'],
    'max_grad_norm':    params['max_grad_norm'],
    '_encoder_width':   encoder_width,
    'log_std_init':     log_std_init,
    'dsr_warmup_steps': params['dsr_warmup_steps'],
    'reward_scale_net': params['reward_scale_net'],
    'reward_scale_dsr': params['reward_scale_dsr'],
    'dsr_clip':         params['dsr_clip'],
}

print(f'Best trial #{best.number}  value={best.value:.4f}  '
      f'({len(study.trials)} total trials)')
print('\nFROZEN_FEATURE_PICKS = ', end=''); pprint(feature_picks)
print('\nSENTIMENT_PICKS = ', end=''); pprint(reddit_picks)
print(f'\nNaN strategy : {reddit_nan_strategy}')
print('\nBEST_PPO_PARAMS = ', end=''); pprint(ppo_params)
print(f'\nBEST_DSR_ETA  = {dsr_eta}')
print(f'BEST_LOOKBACK  = {lookback}')
print(f'BEST_TIMESTEPS = {total_timesteps}')


---
## 21 · Final Run Functions  *(adapted for sentiment)*

`run_final_train_test` now:
1. Builds raw arrays with sentiment via `build_raw_arrays_with_reddit`
2. Passes `n_features = F + R + 1` to `PerAssetFeaturesExtractor`
3. Stores `reddit_picks` and `reddit_nan_strategy` in the returned summary


In [ ]:
def train_and_eval_final(
    closes_raw:         np.ndarray,
    X_transformed:      np.ndarray,
    market_transformed: Optional[np.ndarray],
    avail_mask:         np.ndarray,
    spreads:            np.ndarray,
    asset_names:        List[str],
    train_idx:          np.ndarray,
    test_idx:           np.ndarray,
    dates_index:        pd.DatetimeIndex,
    seed:               int,
    ppo_params:         Dict[str, Any],
    dsr_eta:            float,
    total_timesteps:    int,
    out_dir:            str,
    logger:             logging.Logger,
    lookback:           int,
    refit_train_idx:    Optional[np.ndarray] = None,
    label:              str = 'final',
) -> Dict[str, Any]:
    train_core_idx, es_val_idx = split_train_for_early_stop(
        refit_train_idx if refit_train_idx is not None else train_idx,
        dates_index, es_years=1, min_train_days=252,
    )

    tr_s = max(int(train_core_idx[0]),  lookback - 1)
    tr_e = min(int(train_core_idx[-1]), closes_raw.shape[0] - 2)
    es_s = max(int(es_val_idx[0]),      lookback - 1)
    es_e = min(int(es_val_idx[-1]),     closes_raw.shape[0] - 2)
    te_s = max(int(test_idx[0]),        lookback - 1)
    te_e = min(int(test_idx[-1]),       closes_raw.shape[0] - 2)

    def _env(s, e):
        return OneDayHoldPortfolioEnv(
            closes_raw=closes_raw, feats=X_transformed, market=market_transformed,
            avail_mask=avail_mask, spreads=spreads,
            dsr_eta=dsr_eta, dsr_clip=ppo_params['dsr_clip'],
            dsr_warmup_steps=ppo_params['dsr_warmup_steps'],
            reward_scale_net=ppo_params['reward_scale_net'],
            reward_scale_dsr=ppo_params['reward_scale_dsr'],
            start_idx=s, end_idx=e, lookback=lookback,
        )

    es_cb = EarlyStopSharpeCallback(
        eval_env_fn=lambda: _env(es_s, es_e),
        eval_freq=50_000, patience_evals=4, min_evals=5, verbose=1)
    diag_cb = DiagnosticsCallback(
        eval_every_steps=50_000,
        train_env_fn  =lambda: _env(tr_s, tr_e),
        es_val_env_fn =lambda: _env(es_s, es_e),
        test_env_fn   =lambda: _env(te_s, te_e),
        name=label)

    tb_path = os.path.join(out_dir, f'tb_{label}')
    logger.info(f'[{label.upper()}] '
                f'train_core={dates_index[train_core_idx[0]]}'
                f'..{dates_index[train_core_idx[-1]]} '
                f'es_val={dates_index[es_val_idx[0]]}'
                f'..{dates_index[es_val_idx[-1]]} '
                f'test={dates_index[test_idx[0]]}'
                f'..{dates_index[test_idx[-1]]}')

    model = _build_and_train_ppo(
        vec_train=make_vec_env(lambda: _env(tr_s, tr_e)),
        ppo_params=ppo_params, total_timesteps=total_timesteps,
        es_callback=es_cb, diag_callback=diag_cb, seed=seed, tb_path=tb_path,
    )

    train_metrics = rollout(model, _env(tr_s, tr_e), dates_index=dates_index)
    es_metrics    = rollout(model, _env(es_s, es_e), dates_index=dates_index)
    test_metrics  = rollout(model, _env(te_s, te_e), dates_index=dates_index)
    ew_metrics    = rollout_equal_weight(_env(te_s, te_e))

    logger.info(f'[{label.upper()} EVAL] '
                f"test_sharpe={test_metrics['sharpe']:.4f} "
                f"ann_ret={test_metrics['ann_return']:.4%} "
                f"mdd={test_metrics['max_drawdown']:.4%} "
                f'best_es_sharpe={es_cb.best_sharpe:.4f}')

    model_path = os.path.join(out_dir, f'ppo_{label}_model.zip')
    model.save(model_path)
    save_plots(os.path.join(out_dir, f'plots_{label}'), label,
               test_metrics, ew_metrics, asset_names)
    save_metrics_json(os.path.join(out_dir, f'{label}_test_metrics.json'),  test_metrics)
    save_metrics_json(os.path.join(out_dir, f'{label}_train_metrics.json'), train_metrics)
    save_metrics_json(os.path.join(out_dir, f'{label}_ew_metrics.json'),    ew_metrics)

    return {
        'model': model, 'model_path': model_path,
        'best_es_sharpe': float(es_cb.best_sharpe),
        'train': train_metrics, 'es': es_metrics,
        'test': test_metrics, 'ew_test': ew_metrics,
    }


def run_final_train_test(
    df:                pd.DataFrame,
    assets:            List[str],
    spreads:           np.ndarray,
    feature_picks:     Dict[str, Any],
    reddit_picks:   Dict[str, Any],
    reddit_nan_strategy: str,
    ppo_params:        Dict[str, Any],
    dsr_eta:           float,
    lookback:          int,
    out_dir:           str,
    total_timesteps:   int = 1_000_000,
    seed:              int = SEED_PORTFOLIO_DRL_PPO_V9_1_X_REDDIT_DEFAULT,
    refit_mid_date:    str = '2024-07-01',
    do_refit:          bool = True,
) -> Dict[str, Any]:
    os.makedirs(out_dir, exist_ok=True)
    logger = setup_logger(out_dir, name_prefix='ppo_reddit_final')
    logger.info('=' * 100)
    logger.info('FINAL TRAIN/TEST RUN (REDDIT) START')
    logger.info(f'feature_picks={feature_picks}')
    logger.info(f'reddit_picks={reddit_picks}  NaN strategy={reddit_nan_strategy}')
    logger.info(f'lookback={lookback}  dsr_eta={dsr_eta}  seed={seed}  '
                f'refit={do_refit}@{refit_mid_date}')

    dates_index = pd.DatetimeIndex(df.index)
    train_idx, test_idx = make_final_train_test_split(dates_index)

    reddit_cols = reddit_picks['reddit_cols']
    closes_raw, X_raw, feat_names, market_raw = build_raw_arrays_with_reddit(
        df=df, assets=assets, picks=feature_picks,
        logger=logger, reddit_cols=reddit_cols,
        reddit_nan_strategy=reddit_nan_strategy,
    )
    n_tech_feats = len(get_per_asset_feature_list(feature_picks))
    avail = compute_availability_mask(closes_raw, X_raw, lookback, n_tech_feats)
    n_feats = X_raw.shape[2]   # F + R + 1

    # Fit transforms on initial training window
    fitted, market_ft = fit_transforms_on_train(
        X_raw[train_idx], feat_names,
        market_raw[train_idx] if market_raw is not None else None,
    )
    X_trans, market_trans = apply_transforms(X_raw, fitted, market_raw, market_ft)

    logger.info(f'TRAIN = {dates_index[train_idx[0]]} .. {dates_index[train_idx[-1]]}')
    logger.info(f'TEST  = {dates_index[test_idx[0]]}  .. {dates_index[test_idx[-1]]}')
    logger.info(f'Feature dim: total={n_feats}  tech={n_tech_feats}  '
                f'reddit={n_feats - n_tech_feats - 1}  _REDDIT_NAN=1')

    if do_refit:
        test_first_idx, test_second_idx = make_midtest_refit_split(
            test_idx, dates_index, mid_date=refit_mid_date)
    else:
        test_first_idx  = test_idx
        test_second_idx = None

    # ── Rebuild policy_kwargs with updated n_features ──────────────────────
    has_market_flag  = (feature_picks.get('Market') is not None)
    encoder_width_   = ppo_params.pop('_encoder_width', ppo_params.get('net_arch', [64])[0])
    log_std_init_    = ppo_params.pop('log_std_init', -1.0)
    ppo_params['policy_kwargs'] = dict(
        features_extractor_class=PerAssetFeaturesExtractor,
        features_extractor_kwargs=dict(
            n_assets=len(assets),
            lookback=lookback,
            n_features=n_feats,       # ← includes sentiment + _REDDIT_NAN
            has_market=has_market_flag,
            encoder_width=encoder_width_,
        ),
        net_arch=[encoder_width_, encoder_width_],
        activation_fn=nn.Tanh,
        log_std_init=log_std_init_,
    )

    # ── Phase 1 ────────────────────────────────────────────────────────────
    result1 = train_and_eval_final(
        closes_raw=closes_raw, X_transformed=X_trans,
        market_transformed=market_trans, avail_mask=avail,
        spreads=spreads, asset_names=assets,
        train_idx=train_idx, test_idx=test_first_idx,
        dates_index=dates_index, seed=seed,
        ppo_params=ppo_params, dsr_eta=dsr_eta,
        total_timesteps=total_timesteps, out_dir=out_dir,
        logger=logger, lookback=lookback, label='phase1',
    )

    all_test_nets = [result1['test']['net']]
    all_test_w    = [result1['test']['w']]
    all_ew_nets   = [result1['ew_test']['net']]

    # ── Phase 2 (optional refit) ───────────────────────────────────────────
    if do_refit and test_second_idx is not None:
        refit_train_idx = np.concatenate([train_idx, test_first_idx])
        fitted2, market_ft2 = fit_transforms_on_train(
            X_raw[refit_train_idx], feat_names,
            market_raw[refit_train_idx] if market_raw is not None else None,
        )
        X_trans2, market_trans2 = apply_transforms(X_raw, fitted2, market_raw, market_ft2)
        logger.info(f'REFIT train window up to {dates_index[refit_train_idx[-1]]}')

        result2 = train_and_eval_final(
            closes_raw=closes_raw, X_transformed=X_trans2,
            market_transformed=market_trans2, avail_mask=avail,
            spreads=spreads, asset_names=assets,
            train_idx=train_idx, test_idx=test_second_idx,
            dates_index=dates_index, seed=seed,
            ppo_params=ppo_params, dsr_eta=dsr_eta,
            total_timesteps=total_timesteps, out_dir=out_dir,
            logger=logger, lookback=lookback,
            refit_train_idx=refit_train_idx, label='phase2',
        )
        all_test_nets.append(result2['test']['net'])
        all_test_w.append(result2['test']['w'])
        all_ew_nets.append(result2['ew_test']['net'])
    else:
        result2 = None

    # ── Combine ────────────────────────────────────────────────────────────
    combined_test_nets = np.concatenate(all_test_nets)
    combined_test_w    = np.vstack(all_test_w)
    combined_ew_nets   = np.concatenate(all_ew_nets)

    train_dates = result1['train']['dates']
    test1_dates = result1['test']['dates']
    if result2 is not None:
        test2_dates = result2['test']['dates']
        test_dates  = test1_dates.append(test2_dates)
    else:
        test_dates  = test1_dates

    train_w      = result1['train']['w']
    all_dates    = train_dates.append(test_dates)
    all_weights  = np.vstack([train_w, combined_test_w])

    weights_df = pd.DataFrame(all_weights, index=all_dates, columns=assets)
    weights_df.index.name = 'date'
    train_weights_df = weights_df.loc[train_dates]
    test_weights_df  = weights_df.loc[test_dates]

    summary = {
        'feature_picks':     feature_picks,
        'reddit_picks':   reddit_picks,
        'reddit_nan_strategy': reddit_nan_strategy,
        'feat_names':        feat_names,
        'asset_names':       assets,
        'n_total_feats':     n_feats,
        'n_tech_feats':      n_tech_feats,
        'ppo_params':        ppo_params,
        'dsr_eta':           dsr_eta,
        'lookback':          lookback,
        'total_timesteps':   total_timesteps,
        'seed':              seed,
        'do_refit':          do_refit,
        'refit_mid_date':    refit_mid_date if do_refit else None,
        'train_period':      [str(dates_index[train_idx[0]].date()),
                              str(dates_index[train_idx[-1]].date())],
        'test_period':       [str(dates_index[test_idx[0]].date()),
                              str(dates_index[test_idx[-1]].date())],
        'phase1_test_sharpe':     result1['test']['sharpe'],
        'phase1_test_ann_return': result1['test']['ann_return'],
        'phase1_test_mdd':        result1['test']['max_drawdown'],
        'phase1_best_es_sharpe':  result1['best_es_sharpe'],
        'phase2_test_sharpe':     result2['test']['sharpe'] if result2 else None,
        'phase2_test_ann_return': result2['test']['ann_return'] if result2 else None,
        'phase2_test_mdd':        result2['test']['max_drawdown'] if result2 else None,
        'phase2_best_es_sharpe':  result2['best_es_sharpe'] if result2 else None,
        'combined_test_sharpe':   annualized_sharpe(combined_test_nets),
        'combined_test_ann_return': annualized_return(combined_test_nets),
        'combined_test_mdd':      max_drawdown(combined_test_nets),
        'ew_test_sharpe':         annualized_sharpe(combined_ew_nets),
        'train_weights':      result1['train']['w'].tolist(),
        'test_weights':       combined_test_w.tolist(),
        'all_weights':        all_weights,
        'weights_df':         weights_df,
        'train_weights_df':   train_weights_df,
        'test_weights_df':    test_weights_df,
    }

    logger.info('=' * 100)
    logger.info('FINAL RUN DONE')
    logger.info(f"combined_test_sharpe={summary['combined_test_sharpe']:.4f}  "
                f"ann_ret={summary['combined_test_ann_return']:.4%}  "
                f"mdd={summary['combined_test_mdd']:.4%}")
    if result2:
        logger.info(f"phase1_sharpe={summary['phase1_test_sharpe']:.4f}  "
                    f"phase2_sharpe={summary['phase2_test_sharpe']:.4f}")

    return summary


---
## 22 · Run Final Train / Test

In [ ]:
summary = run_final_train_test(
    df=df,
    assets=ASSETS,
    spreads=spreads,
    feature_picks=feature_picks,
    reddit_picks=reddit_picks,
    reddit_nan_strategy=reddit_nan_strategy,
    ppo_params=ppo_params,
    dsr_eta=dsr_eta,
    lookback=lookback,
    out_dir=f'{OUT_DIR}/final_run',
    total_timesteps=total_timesteps,
    seed=SEED_PORTFOLIO_DRL_PPO_V9_1_X_REDDIT,
    do_refit=True,
    refit_mid_date='2024-07-01',
)

print(f"Combined test Sharpe : {summary['combined_test_sharpe']:.4f}")
print(f"Combined test Ann Ret: {summary['combined_test_ann_return']:.4%}")
print(f"Combined test MDD    : {summary['combined_test_mdd']:.4%}")


In [ ]:
non_tradable_days = str(PROJECT_ROOT / '01_data/aux/non_tradable_days.txt')
must_have_weights = pd.DataFrame(
    index=pd.date_range(start='2023-07-01', end='2025-06-30', freq='D'))
non_tradable_dates = pd.to_datetime(pd.read_csv(non_tradable_days, header=None)[0])
must_have_weights  = must_have_weights.drop(non_tradable_dates, errors='ignore')

weights = must_have_weights.join(summary['test_weights_df'], how='left')
weights = weights.fillna(method='ffill').fillna(method='bfill')[ASSETS].values


---
## 23 · Portfolio Metrics

In [ ]:
import sys
from pathlib import Path
from src.metrics import PortfolioMetrics

pm = PortfolioMetrics()


In [ ]:
pm.summary(weights)

In [ ]:
pm.plot_weights(weights)

In [ ]:
pm.plot_cumulative_returns(weights)

In [ ]:
np.save("DRLPPO_reddit_weights.npy", weights)